# try the not renamed event log instead 

- import libraries:

In [1]:
import pm4py # process mining library
import pandas as pd # data manipulation library
import os # operating system library for file handling
import matplotlib.pyplot as plt # plotting library for visualizations
import numpy as np # numerical computing library for data analysis
import networkx as nx # library for graph analysis and visualization
import math # mathematical functions for calculations

- load only completed and relabled events 

In [2]:
df = pd.read_csv("../data/complete_log.csv", low_memory=False)
df['time:timestamp'] = pd.to_datetime(df['time:timestamp'], format='mixed', utc=True)
event_log = pm4py.format_dataframe(df, case_id='case:concept:name', activity_key='concept:name', timestamp_key='time:timestamp')


### Define own Simplicity metrics

In the book [3], the following metrics are mentioned as candidates for simplicity metrics by analyzing the complexity of the underlying graph:
- Size
- Diameter
- Density
- Connectivity
- Node Degree

Given that the graph functions as a process model, additional domain-specific metrics can be applied:

- Sequentiality: Gateway arcs divided by total arcs.
- Structuredness: Proportion of reducible, well-structured parts in the model.
- Depth: Average or max depth of split/join constructs.
- Gateway Mismatch: Difference between input and output arcs for connected gateways.
- Gateway Heterogeneity: Entropy of gateway types used.
- Control Flow Complexity: Sum of choices based on split types and outgoing arcs.
- Cyclicity: Ratio of nodes in cycles to total nodes.
- Token Splits: Number of concurrent threads from AND/OR-splits.


[3] Josep Carmona, Boudewijn F. van Dongen, Andreas Solti, and Matthias Weidlich. *Conformance Checking - Relating Processes and Models*. Springer, 2018, pp. 223–224.

- a first simple simplicity metrics is the size (arcs and nodes) that is implemented in the following way:

In [3]:
def calculate_node_size(net):
    """
    Calculates the total number of nodes in a pm4py Petri net.
    Nodes = Places + Transitions
    """
    num_places = len(net.places)
    num_transitions = len(net.transitions)
    
    return num_places + num_transitions

def calculate_arc_size(net):
    """
    Calculates the total number of arcs (edges) in a pm4py Petri net.
    """
    return len(net.arcs)

- another metric that makes sense in general is the cyclicity of the net
- implementation:

In [4]:
def calculate_cyclicity(net):
    """
    Calculates the cyclicity of a pm4py Petri net.
    Cyclicity = (number of nodes within cycles) / (total number of nodes)
    
    :param net: pm4py Petri net object
    :return: Float representing the cyclicity metric (between 0.0 and 1.0)
    """
    # 1. Calculate total number of nodes (Places + Transitions)
    total_nodes = len(net.places) + len(net.transitions)
    
    if total_nodes == 0:
        return 0.0
        
    # 2. Build a NetworkX Directed Graph
    G = nx.DiGraph()
    
    # Add all places and transitions as nodes
    for place in net.places:
        G.add_node(place)
    for transition in net.transitions:
        G.add_node(transition)
        
    # Add all arcs as directed edges
    for arc in net.arcs:
        G.add_edge(arc.source, arc.target)
        
    # 3. Identify nodes in cycles
    # A node is in a cycle if it is part of a Strongly Connected Component (SCC)
    # with more than 1 node, or if it is a single node with a self-loop.
    nodes_in_cycles = set()
    
    for scc in nx.strongly_connected_components(G):
        if len(scc) > 1:
            # All nodes in an SCC of size > 1 are part of a cycle
            nodes_in_cycles.update(scc)
        elif len(scc) == 1:
            # Check for self-loops for single-node components
            node = list(scc)[0]
            if G.has_edge(node, node):
                nodes_in_cycles.add(node)
                
    # 4. Calculate the cyclicity ratio
    cyclicity = len(nodes_in_cycles) / total_nodes
    
    return cyclicity

### process discovery 

In [ ]:
import os
import pandas as pd
import pm4py
from pm4py.algo.discovery.heuristics import algorithm as heuristics_miner

output_dir = "../petrinets/heuristic_fine_grid_search_top100"
os.makedirs(output_dir, exist_ok=True)

# 1. Filter log strictly on the top 100 variants
k_variants = 100
print(f"Filtering for top {k_variants} variants...")
filtered_log = pm4py.filter_variants_top_k(event_log, k_variants)

# 2. Define Finer Thresholds for Grid Search 
# Granular steps, especially heavily testing the upper bounds (0.9+)
dependency_thresholds = [0.3, 0.65, 0.9, 0.95, 0.99]
and_thresholds = [0.4, 0.65, 0.9, 0.95, 0.99]
loop_thresholds = [0.3, 0.65, 0.9, 0.95, 0.99]

total_combinations = len(dependency_thresholds) * len(and_thresholds) * len(loop_thresholds)
results_list = []

print(f"Running fine grid search on {total_combinations} combinations...")

for dep in dependency_thresholds:
    for and_thresh in and_thresholds:
        for loop_thresh in loop_thresholds:
            
            # Setup parameters dictionary 
            params = {
                heuristics_miner.Variants.CLASSIC.value.Parameters.DEPENDENCY_THRESH: dep,
                heuristics_miner.Variants.CLASSIC.value.Parameters.AND_MEASURE_THRESH: and_thresh,
                heuristics_miner.Variants.CLASSIC.value.Parameters.LOOP_LENGTH_TWO_THRESH: loop_thresh
            }
            
            # 3. Discover Petri net using the custom parameters
            net, im, fm = heuristics_miner.apply(filtered_log, parameters=params)
            
            # 4. Save artifacts (using :.2f to prevent messy float filenames)
            model_name = f"heu_top100_dep_{dep:.2f}_and_{and_thresh:.2f}_loop_{loop_thresh:.2f}"
            pnml_path = os.path.join(output_dir, f"{model_name}.pnml")
            pdf_path = os.path.join(output_dir, f"{model_name}.pdf")
            
            pm4py.write_pnml(net, im, fm, pnml_path)
            pm4py.save_vis_petri_net(net, im, fm, pdf_path)
            
            # 5. Conformance (Full Log) & Simplicity Metrics
            fitness = pm4py.algo.evaluation.replay_fitness.variants.token_replay.apply(event_log, net, im, fm)
            precision = pm4py.algo.evaluation.precision.variants.etconformance_token.apply(event_log, net, im, fm)
            generalization = pm4py.algo.evaluation.generalization.variants.token_based.apply(event_log, net, im, fm)
            
            # Safely extract perfect fitting traces
            perfect_fit_traces = fitness.get('perc_fit_traces', 0)
            
            results_list.append({
                'Model Name': model_name,
                'Dep Thresh': dep,
                'AND Thresh': and_thresh,
                'Loop Thresh': loop_thresh,
                'Perfect Fitting Traces (%)': round(perfect_fit_traces, 2),
                'Average Trace Fitness': fitness['log_fitness'],
                'Precision': precision,
                'Generalization': generalization,
                'Node Size': calculate_node_size(net),
                'Arc Size': calculate_arc_size(net),
                'Cyclicity': calculate_cyclicity(net)
            })
            print(f"Done: {model_name}")

# 6. Display Results
grid_search_df = pd.DataFrame(results_list)
# Sort to easily see your best performing models at the top
grid_search_df = grid_search_df.sort_values(by=['Average Trace Fitness', 'Precision'], ascending=False)
grid_search_df

Filtering for top 282 variants...
Running fine grid search on 125 combinations...


replaying log with TBR, completed traces ::   0%|          | 0/5623 [00:00<?, ?it/s]

replaying log with TBR, completed traces ::   0%|          | 0/27816 [00:00<?, ?it/s]

replaying log with TBR, completed traces ::   0%|          | 0/5623 [00:00<?, ?it/s]

Done: heu_top282_dep_0.30_and_0.40_loop_0.30


...

replaying log with TBR, completed traces ::   0%|          | 0/5623 [00:00<?, ?it/s]

replaying log with TBR, completed traces ::   0%|          | 0/27816 [00:00<?, ?it/s]

replaying log with TBR, completed traces ::   0%|          | 0/5623 [00:00<?, ?it/s]

Done: heu_top282_dep_0.99_and_0.99_loop_0.99


,Model Name,Dep Thresh,AND Thresh,Loop Thresh,Perfect Fitting Traces (%),Average Trace Fitness,Precision,Generalization,Node Size,Arc Size,Cyclicity
115,heu_top282_dep_0.99_and_0.95_loop_0.30,0.99,0.95,0.30,4.14,0.942761,0.839399,0.94743,87,115,0.689655
116,heu_top282_dep_0.99_and_0.95_loop_0.65,0.99,0.95,0.65,4.14,0.942761,0.839399,0.94743,87,115,0.689655
117,heu_top282_dep_0.99_and_0.95_loop_0.90,0.99,0.95,0.90,4.14,0.942761,0.839399,0.94743,87,115,0.689655
118,heu_top282_dep_0.99_and_0.95_loop_0.95,0.99,0.95,0.95,4.14,0.942761,0.839399,0.94743,87,115,0.689655
119,heu_top282_dep_0.99_and_0.95_loop_0.99,0.99,0.95,0.99,4.14,0.942761,0.839399,0.94743,87,115,0.689655
...,...,...,...,...,...,...,...,...,...,...,...
75,heu_top282_dep_0.95_and_0.40_loop_0.30,0.95,0.40,0.30,0.00,0.832066,0.999146,0.92867,91,124,0.670330
76,heu_top282_dep_0.95_and_0.40_loop_0.65,0.95,0.40,0.65,0.00,0.832066,0.999146,0.92867,91,124,0.670330
77,heu_top282_dep_0.95_and_0.40_loop_0.90,0.95,0.40,0.90,0.00,0.832066,0.999146,0.92867,91,124,0.670330
78,heu_top282_dep_0.95_and_0.40_loop_0.95,0.95,0.40,0.95,0.00,0.832066,0.999146,0.92867,91,124,0.670330


In [6]:
grid_search_df.to_csv(os.path.join(output_dir, "heuristic_grid_search_results_top282.csv"), index=False)